<font color="red" size="10">
<b>DEMO TIME: 15 mins</b>
</font>

# About this Notebook

## Generating Agent Trajectories for ART and RULER

This notebook demonstrates different approaches to **generating multiple agent trajectories** to prepare agents for the optimization stage in Agent Reinforcement Training (ART).

Trajectories are sequences of agent interactions (messages, tool calls, responses) that serve as training data. This notebook shows:
- Automated scenario generation for MCP servers
- Manual trajectory creation
- Synthetic trajectory generation
- Real-world trajectory collection
- How trajectories are used with RULER for relative ranking


This notebook is for the *Demo* for Session 3 for Develop Self-Improving AI Agents with Reinforcement Learning Live Event with O'Reilly Media by
[Nicole Koenigstein](https://www.linkedin.com/in/nicole-koenigstein/).


# Additional Resources


<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>


In [1]:
import codecs
from IPython.display import Markdown, display

def reveal_answer(which: str = "", key: str = ""):
    answers = {
        "q1": "Lbh arrq gur shapgvbaf naq gur genvavat ybbc. Gur gbby pnyyf naq erfhygf ner \ngur npgvbaf. Jvgubhg gurz, lbh pna'g qb perqvg nffvtazrag, naq EY pna'g xabj jung vg fubhyq erjneq.",
        "q2": "Vg arrqf zhygvcyr genwrpgbevrf sbe gur fnzr fpranevb gb enax jvguva n tebhc. Vs rirel genwrpgbel vf n qvssrerag gnfx, gur enaxvat fvtany vf qbzvangrq ol qvssvphygl naq orpbzrf abg hfrshy sbe EY.",
    }

    if key.strip().lower() == "decode":
        secret = answers.get(which, "")
        decoded = codecs.decode(secret, "rot_13")
        display(Markdown("### **Decoded Answer**\n\n<font size='5' color='green'>" + decoded + "</font>"))
    else:
        display(Markdown("### **Answer remains encoded.**\n\nProvide key = 'decode' to reveal."))


## Installation


In [2]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install openpipe-art[backend]==0.4.11 tenacity "mcp>=1.11.0" "gql<4" aiohttp --prerelease allow --no-cache-dir
else:
    try:
        import numpy
        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    get_vllm, get_triton = (
        ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    )
    !uv pip install --upgrade \
        openpipe-art[backend]==0.4.11 tenacity pillow==11.3.0 protobuf==5.29.5 {get_vllm} {get_numpy} --prerelease allow --no-cache-dir
    !uv pip install -qqq {get_triton}


In [3]:
import os
import random
import time
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

from dotenv import load_dotenv

import art
from art.trajectories import Trajectory, Choice, TrajectoryGroup
from art.mcp import generate_scenarios
from art.mcp.generate_scenarios import preview_scenarios
from art.utils.logging import info, ok, step, warn, err

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
EXA_API_KEY = os.getenv('EXA_API_KEY')

# Initialize variables that might be used later
mcp_scenarios = []

print("ART version:", art.__version__ if hasattr(art, '__version__') else "installed")


ART version: installed


## What are Agent Trajectories?

In ART, a **trajectory** represents a complete interaction sequence between an agent and its environment:

- **Messages**: User inputs, system prompts, assistant responses
- **Tool Calls**: Function invocations with arguments
- **Tool Results**: Responses from tool executions
- **Rewards**: Scores from RULER or other evaluators
- **Metrics**: Metadata about the interaction (turns, success, etc.)

Multiple trajectories are collected and grouped, then RULER performs **relative ranking** to determine which trajectories are better, enabling the model to learn from comparisons rather than absolute scores.


## Approach 1: Automated Scenario Generation (MCP Servers)

For MCP (Model Context Protocol) servers, ART provides `generate_scenarios()` which automatically creates diverse task scenarios based on available tools and resources. This is ideal when you have a well-defined tool set.


In [4]:
# @title MCP Helpers Setup

from contextlib import asynccontextmanager
from mcp.client.session import ClientSession
from mcp.client.streamable_http import streamablehttp_client

EXA_MCP_URL = "https://mcp.exa.ai/mcp"

if not EXA_API_KEY:
    warn("EXA_API_KEY not set. MCP examples will be skipped.")
    EXA_MCP_URL = None

if EXA_MCP_URL and EXA_API_KEY:
    HEADERS = {"Authorization": f"Bearer {EXA_API_KEY}"}

    @asynccontextmanager
    async def mcp_session():
        async with streamablehttp_client(EXA_MCP_URL, headers=HEADERS) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

    async def list_tools_and_resources():
        async with mcp_session() as session:
            tools = await session.list_tools()
            try:
                resources = await session.list_resources()
            except Exception:
                class _Empty:
                    resources = []
                resources = _Empty()
            return tools, resources

    ok("MCP helpers configured")
else:
    warn("MCP helpers not configured - skipping MCP examples")


[15:35:19] OK    MCP helpers configured


# Question 1


<font color="red" size="10">
<b>Question to ALL:</b>
</font>

<br><br>

<font color="blue" size="5">
<b>
If you only log the final assistant answer (and not the tool calls + tool results + intermediate messages),
can you still train an agent reliably with ART and RULER? Why or why not? Type your answer in the Q & A section. </b>
</font>



In [5]:
#reveal_answer("q1", "decode")


In [6]:
# @title Approach 1: Generate Scenarios for MCP Server

if EXA_MCP_URL and EXA_API_KEY and OPENROUTER_API_KEY:
    info("Generating scenarios using MCP tools...")

    # Get available tools and resources
    tools_result, resources_result = await list_tools_and_resources()

    tools_list = [
        {"name": t.name, "description": t.description, "parameters": t.inputSchema}
        for t in (tools_result.tools or [])
    ]

    resources_list = [
        {
            "uri": str(r.uri),
            "name": r.name,
            "description": r.description,
            "mimeType": r.mimeType,
        }
        for r in (getattr(resources_result, "resources", []) or [])
    ]

    info(f"Available: {len(tools_list)} tool(s), {len(resources_list)} resource(s).")

    # Generate scenarios
    num_scenarios = 10  # Small number for demonstration

    try:
        scenario_collection = await generate_scenarios(
            tools=tools_list,
            resources=resources_list,
            num_scenarios=num_scenarios,
            show_preview=False,
            generator_model="openai/o4-mini",
            generator_api_key=OPENROUTER_API_KEY,
        )

        scenarios = [{"task": s.task, "difficulty": s.difficulty} for s in scenario_collection.scenarios]

        ok(f"Generated {len(scenarios)} scenarios")

        info("Sample scenarios:")
        preview_scenarios(scenarios, n=min(5, len(scenarios)))

        # Store for later use
        mcp_scenarios = scenarios

    except Exception as e:
        warn(f"Scenario generation failed: {e}")
        mcp_scenarios = []
else:
    warn("Skipping MCP scenario generation (missing API keys)")
    mcp_scenarios = []


[15:35:19] INFO  Generating scenarios using MCP tools...
[15:35:21] INFO  Available: 2 tool(s), 1 resource(s).
[15:35:21] OK    Using model: openai/o4-mini
[15:35:21] INFO  Available: 2 tool(s), 1 resource(s).
[15:35:21] STEP  Preparing prompt & JSON schema &
[15:35:21] STEP  Calling model: openai/o4-mini &
[15:35:41] OK    Model responded in 19.66s.
[15:35:41] INFO  Raw content length: 2565 chars.
[15:35:41] OK    Parsed 10 scenario(s) successfully.
[15:35:41] INFO  Difficulty distribution:
   1/5:   2  ██
   2/5:   1  █
   3/5:   4  ████
   4/5:   2  ██
   5/5:   1  █
[15:35:41] OK    Generated 10 scenarios in 19.73s total.
[15:35:41] OK    Generated 10 scenarios
[15:35:41] INFO  Sample scenarios:
   1. Summarize the top 5 most recent news articles about AI regulation from reputable tech and policy websites, providing a c&  (difficulty 1/5)
   2. Provide a set of example code snippets and best practices for using the Python requests library to perform GET and POST&  (difficulty 1/5)


### Benefits of Automated Scenario Generation:

✅ **Diverse tasks**: Automatically creates varied scenarios based on tool capabilities
✅ **No manual work**: No need to write individual test cases
✅ **Tool-aware**: Scenarios are tailored to available tools and resources
✅ **Scalable**: Generate hundreds of scenarios quickly

**Use when**: You have a well-defined MCP server with clear tools and resources


## Approach 2: Manual Trajectory Creation

For specific use cases or when you need precise control, you can manually create trajectories. This is useful for:
- Prototyping new agent behaviors
- Creating golden examples
- Testing specific failure modes


In [7]:
# @title Approach 2: Manual Trajectory Creation

# Define tools for the game
game_tools = [
    {
        "type": "function",
        "function": {
            "name": "make_move",
            "description": "Make a move in rock paper scissors",
            "parameters": {
                "type": "object",
                "properties": {
                    "move": {
                        "type": "string",
                        "enum": ["rock", "paper", "scissors"],
                        "description": "Your move",
                    }
                },
                "required": ["move"],
            },
        },
    }
]

# -----------------------------
# Good trajectory (agent wins)
# -----------------------------
good_trajectory = Trajectory(
    messages_and_choices=[
        {"role": "system", "content": "You are playing rock paper scissors. Make strategic moves."},
        {"role": "user", "content": "Let's play! I'll go first. I choose rock."},

        # Choice: assistant triggers a tool call
        {
            "finish_reason": "tool_calls",
            "index": 0,
            "message": {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": "call_1",
                        "type": "function",
                        "function": {"name": "make_move", "arguments": '{"move": "paper"}'},
                    }
                ],
            },
        },

        # Tool response (must reference tool_call_id == tool_calls[0].id)
        {"role": "tool", "tool_call_id": "call_1", "content": "You win! Paper beats rock."},

        # Choice: assistant final natural language
        {
            "finish_reason": "stop",
            "index": 1,
            "message": {
                "role": "assistant",
                "content": "Great! I chose paper, which beats your rock. Good game!",
            },
        },
    ],
    tools=game_tools,
    reward=0.9,

    # metrics: numeric or bool only
    metrics={
        "game_id": 1,          # 1 = rock paper scissors
        "won": 1,              # bool is allowed too, but 1 is fine
        "moves": 1,
        "strategy_id": 1,      # 1 = counter
    },

    # metadata: strings allowed
    metadata={
        "game": "rock_paper_scissors",
        "result": "win",
        "strategy": "counter_opponent",
    },
)

# -----------------------------
# Poor trajectory (agent loses)
# -----------------------------
poor_trajectory = Trajectory(
    messages_and_choices=[
        {"role": "system", "content": "You are playing rock paper scissors. Make strategic moves."},
        {"role": "user", "content": "Let's play! I'll go first. I choose paper."},

        {
            "finish_reason": "tool_calls",
            "index": 0,
            "message": {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": "call_1",
                        "type": "function",
                        "function": {"name": "make_move", "arguments": '{"move": "rock"}'},
                    }
                ],
            },
        },

        {"role": "tool", "tool_call_id": "call_1", "content": "You lose! Paper beats rock."},

        {
            "finish_reason": "stop",
            "index": 1,
            "message": {
                "role": "assistant",
                "content": "Oh no, I chose rock and you chose paper. You win this round!",
            },
        },
    ],
    tools=game_tools,
    reward=0.2,
    metrics={
        "game_id": 1,
        "won": 0,
        "moves": 1,
        "strategy_id": 0,      # 0 = random
    },
    metadata={
        "game": "rock_paper_scissors",
        "result": "lose",
        "strategy": "random",
    },
)

print("Approach 2: Manual Trajectory Creation")
print("=" * 60)
print(f"\nGood trajectory (reward: {good_trajectory.reward}):")
print(f"  Messages: {len(good_trajectory.messages_and_choices)}")
print(f"  Result: {good_trajectory.metadata.get('result')}")
print(f"\nPoor trajectory (reward: {poor_trajectory.reward}):")
print(f"  Messages: {len(poor_trajectory.messages_and_choices)}")
print(f"  Result: {poor_trajectory.metadata.get('result')}")

manual_trajectories = [good_trajectory, poor_trajectory]


Approach 2: Manual Trajectory Creation

Good trajectory (reward: 0.9):
  Messages: 5
  Result: win

Poor trajectory (reward: 0.2):
  Messages: 5
  Result: lose


In [8]:
import json

def clean_print_trajectory(t: Trajectory):
    print("=" * 80)
    print(f"Game: {t.metadata.get('game')}")
    print(f"Reward: {t.reward}")
    print("Metrics:")
    for k, v in t.metrics.items():
        print(f"  {k}: {v}")

    print("\nTrajectory Messages:\n")

    for i, item in enumerate(t.messages_and_choices):
        # Case 1: regular chat messages are dicts
        if isinstance(item, dict):
            role = item.get("role")
            print(f"[Turn {i}] ROLE: {role}")
            content = item.get("content")
            if content is not None:
                print(f"  CONTENT: {content}")

            # Tool calls can appear in assistant dict messages too (rare here)
            tool_calls = item.get("tool_calls")
            if tool_calls:
                print("  TOOL CALLS:")
                for call in tool_calls:
                    fn = call["function"]["name"]
                    args = call["function"]["arguments"]
                    print(f"    -> {fn} args={args}")

            # Tool message extra
            if role == "tool":
                print(f"  TOOL_CALL_ID: {item.get('tool_call_id')}")

            continue

        # Case 2: Choice objects (pydantic)
        # item: Choice
        print(f"[Turn {i}] ROLE: assistant (choice)")
        print(f"  FINISH_REASON: {item.finish_reason}")
        print(f"  INDEX: {item.index}")

        msg = item.message  # ChatCompletionMessage (pydantic)

        # Content
        if getattr(msg, "content", None) is not None:
            print(f"  CONTENT: {msg.content}")

        # Tool calls
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            print("  TOOL CALLS:")
            for call in tool_calls:
                # call is ChatCompletionMessageToolCall (pydantic)
                fn = call.function.name
                args = call.function.arguments
                print(f"    -> {fn} args={args} (id={call.id})")

    print("=" * 80)


print("\nGOOD TRAJECTORY:")
clean_print_trajectory(good_trajectory)

print("\nPOOR TRAJECTORY:")
clean_print_trajectory(poor_trajectory)



GOOD TRAJECTORY:
Game: rock_paper_scissors
Reward: 0.9
Metrics:
  game_id: 1
  won: 1
  moves: 1
  strategy_id: 1

Trajectory Messages:

[Turn 0] ROLE: system
  CONTENT: You are playing rock paper scissors. Make strategic moves.
[Turn 1] ROLE: user
  CONTENT: Let's play! I'll go first. I choose rock.
[Turn 2] ROLE: assistant (choice)
  FINISH_REASON: tool_calls
  INDEX: 0
  TOOL CALLS:
    -> make_move args={"move": "paper"} (id=call_1)
[Turn 3] ROLE: tool
  CONTENT: You win! Paper beats rock.
  TOOL_CALL_ID: call_1
[Turn 4] ROLE: assistant (choice)
  FINISH_REASON: stop
  INDEX: 1
  CONTENT: Great! I chose paper, which beats your rock. Good game!

POOR TRAJECTORY:
Game: rock_paper_scissors
Reward: 0.2
Metrics:
  game_id: 1
  won: 0
  moves: 1
  strategy_id: 0

Trajectory Messages:

[Turn 0] ROLE: system
  CONTENT: You are playing rock paper scissors. Make strategic moves.
[Turn 1] ROLE: user
  CONTENT: Let's play! I'll go first. I choose paper.
[Turn 2] ROLE: assistant (choice)
  FIN

In [9]:
from art import Trajectory
print(Trajectory.model_json_schema())


{'$defs': {'Annotation': {'additionalProperties': True, 'properties': {'type': {'const': 'url_citation', 'title': 'Type', 'type': 'string'}, 'url_citation': {'$ref': '#/$defs/AnnotationURLCitation'}}, 'required': ['type', 'url_citation'], 'title': 'Annotation', 'type': 'object'}, 'AnnotationURLCitation': {'additionalProperties': True, 'properties': {'end_index': {'title': 'End Index', 'type': 'integer'}, 'start_index': {'title': 'Start Index', 'type': 'integer'}, 'title': {'title': 'Title', 'type': 'string'}, 'url': {'title': 'Url', 'type': 'string'}}, 'required': ['end_index', 'start_index', 'title', 'url'], 'title': 'AnnotationURLCitation', 'type': 'object'}, 'Audio': {'properties': {'id': {'title': 'Id', 'type': 'string'}}, 'required': ['id'], 'title': 'Audio', 'type': 'object'}, 'ChatCompletionAssistantMessageParam': {'properties': {'role': {'const': 'assistant', 'title': 'Role', 'type': 'string'}, 'audio': {'anyOf': [{'$ref': '#/$defs/Audio'}, {'type': 'null'}]}, 'content': {'anyO

### Benefits of Manual Creation:

✅ **Precise control**: Create exactly the trajectories you need
✅ **Golden examples**: Define ideal agent behavior
✅ **Failure modes**: Test specific error cases
✅ **Prototyping**: Quick iteration on new agent designs

**Use when**: You need specific examples, are prototyping, or want to test edge cases


## Approach 3: Synthetic Trajectory Generation

Generate trajectories programmatically using templates, patterns, or LLM-based generation. This is useful for:
- Creating large datasets quickly
- Ensuring coverage of specific patterns
- Generating variations of existing trajectories


In [10]:
# @title Approach 3: Synthetic Trajectory Generation

def generate_qa_trajectory(question: str, answer: str, quality: str = "good") -> Trajectory:
    """Generate a Q and A trajectory with varying quality."""

    if quality == "good":
        response = f"{answer} This is a comprehensive and accurate answer."
        reward = 0.85
        quality_id = 2
    elif quality == "medium":
        response = f"{answer} This answer is somewhat helpful but could be more detailed."
        reward = 0.60
        quality_id = 1
    else:  # poor
        response = "I'm not entirely sure, but I think the answer might be related to that topic."
        reward = 0.30
        quality_id = 0

    return Trajectory(
        messages_and_choices=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question},

            # Assistant response must be a Choice shaped object
            {
                "finish_reason": "stop",
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": response,
                },
            },
        ],
        reward=reward,

        # metrics must be numeric or bool only
        metrics={
            "task_id": 1,  # 1 = qa
            "quality_id": quality_id,
            "question_length": len(question),
            "answer_length": len(response),
        },

        # metadata can carry strings
        metadata={
            "task": "qa",
            "quality": quality,
        },
    )

# Generate multiple trajectories with variations
qa_pairs = [
    ("What is machine learning?", "Machine learning is a subset of artificial intelligence that enables systems to learn from data."),
    ("How does neural network training work?", "Neural network training involves forward propagation, loss calculation, and backpropagation to update weights."),
    ("What is reinforcement learning?", "Reinforcement learning is a type of machine learning where agents learn through trial and error using rewards."),
]

synthetic_trajectories = []
for question, answer in qa_pairs:
    for quality in ["good", "medium", "poor"]:
        traj = generate_qa_trajectory(question, answer, quality)
        synthetic_trajectories.append(traj)

print("Approach 3: Synthetic Trajectory Generation")
print("=" * 60)
print(f"Generated {len(synthetic_trajectories)} trajectories")

print("\nBreakdown by quality:")
for quality in ["good", "medium", "poor"]:
    subset = [t for t in synthetic_trajectories if t.metadata.get("quality") == quality]
    count = len(subset)
    avg_reward = sum(t.reward for t in subset) / count if count else 0
    print(f"  {quality}: {count} trajectories (avg reward: {avg_reward:.2f})")

print("\nSample trajectory:")
sample = synthetic_trajectories[0]
print(f"  Question: {sample.messages_and_choices[1]['content'][:60]}...")
print(f"  Quality: {sample.metadata.get('quality')}")
print(f"  Reward: {sample.reward}")


Approach 3: Synthetic Trajectory Generation
Generated 9 trajectories

Breakdown by quality:
  good: 3 trajectories (avg reward: 0.85)
  medium: 3 trajectories (avg reward: 0.60)
  poor: 3 trajectories (avg reward: 0.30)

Sample trajectory:
  Question: What is machine learning?...
  Quality: good
  Reward: 0.85


In [11]:
def clean_print_trajectory(t: Trajectory):
    print("=" * 80)
    print(f"Game: {t.metadata.get('game')}")
    print(f"Reward: {t.reward}")

    print("Metrics:")
    for k, v in t.metrics.items():
        print(f"  {k}: {v}")

    print("\nTrajectory Messages:\n")

    for i, item in enumerate(t.messages_and_choices):
        if isinstance(item, dict):
            role = item.get("role")
            print(f"[Turn {i}] ROLE: {role}")
            content = item.get("content")
            if content is not None:
                print(f"  CONTENT: {content}")
            if item.get("tool_calls"):
                print("  TOOL CALLS:")
                for call in item["tool_calls"]:
                    print(f"    -> {call['function']['name']} args={call['function']['arguments']}")
            continue

        print(f"[Turn {i}] ROLE: assistant")
        print(f"  FINISH_REASON: {item.finish_reason}")
        print(f"  INDEX: {item.index}")

        msg = item.message
        if msg.content is not None:
            print(f"  CONTENT: {msg.content}")

        if msg.tool_calls:
            print("  TOOL CALLS:")
            for call in msg.tool_calls:
                print(f"    -> {call.function.name} args={call.function.arguments} (id={call.id})")

    print("=" * 80)


In [12]:
print("\n--- PRINTING ALL SYNTHETIC QA TRAJECTORIES ---\n")

for i, t in enumerate(synthetic_trajectories):
    print(f"\nSynthetic QA Trajectory #{i}\n")
    clean_print_trajectory(t)



--- PRINTING ALL SYNTHETIC QA TRAJECTORIES ---


Synthetic QA Trajectory #0

Game: None
Reward: 0.85
Metrics:
  task_id: 1
  quality_id: 2
  question_length: 25
  answer_length: 141

Trajectory Messages:

[Turn 0] ROLE: system
  CONTENT: You are a helpful assistant.
[Turn 1] ROLE: user
  CONTENT: What is machine learning?
[Turn 2] ROLE: assistant
  FINISH_REASON: stop
  INDEX: 0
  CONTENT: Machine learning is a subset of artificial intelligence that enables systems to learn from data. This is a comprehensive and accurate answer.

Synthetic QA Trajectory #1

Game: None
Reward: 0.6
Metrics:
  task_id: 1
  quality_id: 1
  question_length: 25
  answer_length: 156

Trajectory Messages:

[Turn 0] ROLE: system
  CONTENT: You are a helpful assistant.
[Turn 1] ROLE: user
  CONTENT: What is machine learning?
[Turn 2] ROLE: assistant
  FINISH_REASON: stop
  INDEX: 0
  CONTENT: Machine learning is a subset of artificial intelligence that enables systems to learn from data. This answer is somewhat

In [13]:
print("\n--- PRINTING GAME TRAJECTORIES TOO ---\n")

print("\nGOOD GAME TRAJECTORY:")
clean_print_trajectory(good_trajectory)

print("\nPOOR GAME TRAJECTORY:")
clean_print_trajectory(poor_trajectory)



--- PRINTING GAME TRAJECTORIES TOO ---


GOOD GAME TRAJECTORY:
Game: rock_paper_scissors
Reward: 0.9
Metrics:
  game_id: 1
  won: 1
  moves: 1
  strategy_id: 1

Trajectory Messages:

[Turn 0] ROLE: system
  CONTENT: You are playing rock paper scissors. Make strategic moves.
[Turn 1] ROLE: user
  CONTENT: Let's play! I'll go first. I choose rock.
[Turn 2] ROLE: assistant
  FINISH_REASON: tool_calls
  INDEX: 0
  TOOL CALLS:
    -> make_move args={"move": "paper"} (id=call_1)
[Turn 3] ROLE: tool
  CONTENT: You win! Paper beats rock.
[Turn 4] ROLE: assistant
  FINISH_REASON: stop
  INDEX: 1
  CONTENT: Great! I chose paper, which beats your rock. Good game!

POOR GAME TRAJECTORY:
Game: rock_paper_scissors
Reward: 0.2
Metrics:
  game_id: 1
  won: 0
  moves: 1
  strategy_id: 0

Trajectory Messages:

[Turn 0] ROLE: system
  CONTENT: You are playing rock paper scissors. Make strategic moves.
[Turn 1] ROLE: user
  CONTENT: Let's play! I'll go first. I choose paper.
[Turn 2] ROLE: assistant
  FI

### Benefits of Synthetic Generation:

✅ **Scalability**: Generate thousands of trajectories quickly
✅ **Coverage**: Ensure specific patterns are represented
✅ **Variations**: Create multiple versions of the same scenario
✅ **Controlled quality**: Generate good/bad examples systematically

**Use when**: You need large datasets, want to test specific patterns, or need systematic variations


## Approach 4: Real-World Trajectory Collection

Collect trajectories from actual agent interactions in production or during development. This captures real user behavior and agent responses.


In [14]:
# @title Approach 4: Real-World Trajectory Collection (Fixed)

from typing import Any, Dict, Tuple

TASK_TO_ID = {
    "weather_query": 1,
    "search": 2,
    "code_generation": 3,
}

SATISFACTION_TO_ID = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

def split_metrics_and_metadata(metrics: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Keep only numeric or bool values in metrics.
    Move strings (and anything else non numeric non bool) into metadata.
    Also add numeric encodings for common categorical fields when present.
    """
    clean_metrics: Dict[str, Any] = {}
    moved_metadata: Dict[str, Any] = {}

    for k, v in (metrics or {}).items():
        if isinstance(v, (int, float, bool)):
            clean_metrics[k] = v
        else:
            moved_metadata[k] = v

    # Optional: encode categories to numeric metrics for easier plotting
    task = moved_metadata.get("task")
    if isinstance(task, str):
        clean_metrics["task_id"] = TASK_TO_ID.get(task, 0)

    satisfaction = moved_metadata.get("user_satisfaction")
    if isinstance(satisfaction, str):
        clean_metrics["user_satisfaction_id"] = SATISFACTION_TO_ID.get(satisfaction, -1)

    return clean_metrics, moved_metadata


def collect_trajectory_from_log(log_entry: Dict[str, Any]) -> Trajectory:
    metrics_in = log_entry.get("metrics", {}) or {}
    clean_metrics, moved = split_metrics_and_metadata(metrics_in)

    metadata = dict(log_entry.get("metadata", {}) or {})
    # Merge moved fields into metadata without losing existing keys
    for k, v in moved.items():
        if k not in metadata:
            metadata[k] = v

    return Trajectory(
        messages_and_choices=log_entry.get("messages", []),
        tools=log_entry.get("tools", None),
        reward=log_entry.get("reward", 0.0),
        metrics=clean_metrics,
        metadata=metadata,
    )

# Simulated production logs
production_logs = [
    {
        "messages": [
            {"role": "user", "content": "What's the weather today?"},
            {"role": "assistant", "content": "I don't have access to weather data. Would you like me to help you find a weather service?"}
        ],
        "reward": 0.5,
        "metrics": {"task": "weather_query", "tool_available": False, "user_satisfaction": "medium"},
        "metadata": {"timestamp": "2024-01-15T10:30:00Z", "session_id": "abc123"}
    },
    {
        "messages": [
            {"role": "user", "content": "Search for Python tutorials"},
            {"role": "assistant", "content": "I found several Python tutorials. Here are the top results: [list of tutorials]"}
        ],
        "tools": [{"type": "function", "function": {"name": "search", "description": "Search the web"}}],
        "reward": 0.8,
        "metrics": {"task": "search", "tool_used": True, "results_found": 5, "user_satisfaction": "high"},
        "metadata": {"timestamp": "2024-01-15T10:35:00Z", "session_id": "abc124"}
    },
    {
        "messages": [
            {"role": "user", "content": "Write a Python function to sort a list"},
            {"role": "assistant", "content": "Here's a Python function to sort a list:\n\n```python\ndef sort_list(items):\n    return sorted(items)\n```"}
        ],
        "reward": 0.9,
        "metrics": {"task": "code_generation", "code_provided": True, "user_satisfaction": "high"},
        "metadata": {"timestamp": "2024-01-15T10:40:00Z", "session_id": "abc125"}
    }
]

real_world_trajectories = [collect_trajectory_from_log(log) for log in production_logs]

print("Approach 4: Real-World Trajectory Collection")
print("=" * 60)
print(f"Collected {len(real_world_trajectories)} trajectories from production logs")

print("\nBreakdown by task:")
task_counts = {}
for traj in real_world_trajectories:
    task = traj.metadata.get("task", "unknown")  # task is now in metadata
    task_counts[task] = task_counts.get(task, 0) + 1
for task, count in task_counts.items():
    print(f"  {task}: {count} trajectories")

avg_reward = sum(t.reward for t in real_world_trajectories) / len(real_world_trajectories)
print(f"\nAverage reward: {avg_reward:.2f}")

print("\nSample trajectory:")
sample = real_world_trajectories[0]
print(f"  Task: {sample.metadata.get('task')}")
print(f"  Reward: {sample.reward}")
print(f"  User message: {sample.messages_and_choices[0]['content'][:60]}...")


Approach 4: Real-World Trajectory Collection
Collected 3 trajectories from production logs

Breakdown by task:
  weather_query: 1 trajectories
  search: 1 trajectories
  code_generation: 1 trajectories

Average reward: 0.73

Sample trajectory:
  Task: weather_query
  Reward: 0.5
  User message: What's the weather today?...


In [15]:

for i, t in enumerate(real_world_trajectories):
    print(f"\nSynthetic QA Trajectory #{i}\n")
    clean_print_trajectory(t)



Synthetic QA Trajectory #0

Game: None
Reward: 0.5
Metrics:
  tool_available: False
  task_id: 1
  user_satisfaction_id: 1

Trajectory Messages:

[Turn 0] ROLE: user
  CONTENT: What's the weather today?
[Turn 1] ROLE: assistant
  CONTENT: I don't have access to weather data. Would you like me to help you find a weather service?

Synthetic QA Trajectory #1

Game: None
Reward: 0.8
Metrics:
  tool_used: True
  results_found: 5
  task_id: 2
  user_satisfaction_id: 2

Trajectory Messages:

[Turn 0] ROLE: user
  CONTENT: Search for Python tutorials
[Turn 1] ROLE: assistant
  CONTENT: I found several Python tutorials. Here are the top results: [list of tutorials]

Synthetic QA Trajectory #2

Game: None
Reward: 0.9
Metrics:
  code_provided: True
  task_id: 3
  user_satisfaction_id: 2

Trajectory Messages:

[Turn 0] ROLE: user
  CONTENT: Write a Python function to sort a list
[Turn 1] ROLE: assistant
  CONTENT: Here's a Python function to sort a list:

```python
def sort_list(items):
    retur

### Benefits of Real-World Collection:

✅ **Authentic data**: Captures real user behavior and agent responses
✅ **Edge cases**: Includes unexpected scenarios from production
✅ **User patterns**: Reflects actual usage patterns
✅ **Continuous improvement**: Can collect data over time

**Use when**: You have a deployed system, want to improve on real usage, or need to capture production edge cases


## Approach 5: Hybrid: Combining Multiple Methods

In practice, you'll often combine multiple approaches to create a comprehensive dataset. This gives you the benefits of each method.


In [16]:
# @title Approach 5: Hybrid Approach

# Combine trajectories from different sources
all_trajectories = []

# Add manual trajectories (golden examples)
all_trajectories.extend(manual_trajectories)
print(f"Added {len(manual_trajectories)} manual trajectories")

# Add synthetic trajectories (systematic coverage)
all_trajectories.extend(synthetic_trajectories)
print(f"Added {len(synthetic_trajectories)} synthetic trajectories")

# Add real-world trajectories (production data)
all_trajectories.extend(real_world_trajectories)
print(f"Added {len(real_world_trajectories)} real-world trajectories")

# If MCP scenarios were generated, we could create trajectories from them too
if mcp_scenarios:
    print(f"\nNote: {len(mcp_scenarios)} MCP scenarios available for trajectory generation")

print("\n" + "=" * 60)
print(f"Total trajectories: {len(all_trajectories)}")
print(f"Average reward: {sum(t.reward for t in all_trajectories) / len(all_trajectories):.2f}")

# Analyze distribution
print("\nReward distribution:")
rewards = [t.reward for t in all_trajectories]
print(f"  Min: {min(rewards):.2f}")
print(f"  Max: {max(rewards):.2f}")
print(f"  Mean: {sum(rewards) / len(rewards):.2f}")
print(f"  Median: {sorted(rewards)[len(rewards) // 2]:.2f}")

# Count trajectories with tools
with_tools = sum(1 for t in all_trajectories if t.tools)
print(f"\nTrajectories with tools: {with_tools} / {len(all_trajectories)}")


Added 2 manual trajectories
Added 9 synthetic trajectories
Added 3 real-world trajectories

Note: 10 MCP scenarios available for trajectory generation

Total trajectories: 14
Average reward: 0.61

Reward distribution:
  Min: 0.20
  Max: 0.90
  Mean: 0.61
  Median: 0.60

Trajectories with tools: 3 / 14


## How Trajectories Are Used with RULER

Once you have trajectories, they're grouped and RULER performs **relative ranking** to determine which trajectories are better. This is more effective than absolute scoring because:

1. **Relative comparisons are easier**: "Is A better than B?" is often clearer than "Rate A on a scale of 1-10"
2. **Consistent ranking**: Judges are more consistent when comparing than when assigning absolute scores
3. **Better signal**: Relative rankings provide clearer learning signal for the model

### Trajectory Groups

Trajectories are organized into **groups** where each group contains multiple trajectories for the same scenario. RULER ranks trajectories within each group.




---



# Question 2


<font color="red" size="10">
<b>Question to ALL: </b>
</font>

<br> <br>
<font color="blue" size="5">
<b>Why does RULER want multiple trajectories for the *same* scenario in a TrajectoryGroup?
What breaks if every trajectory is from a different task? Type your answer in the Q & A section.</b>
</font>



In [17]:
#reveal_answer("q2", "decode")

In [18]:
# @title Creating Trajectory Groups for RULER

# Example: Create groups from synthetic trajectories
# Group trajectories by question (same scenario, different quality responses)
qa_groups = {}
for traj in synthetic_trajectories:
    question = traj.messages_and_choices[1]['content']
    if question not in qa_groups:
        qa_groups[question] = []
    qa_groups[question].append(traj)

# Create TrajectoryGroup objects
trajectory_groups = []
for question, trajs in qa_groups.items():
    group = TrajectoryGroup(trajs)
    trajectory_groups.append(group)

print("Trajectory Groups for RULER")
print("=" * 60)
print(f"Created {len(trajectory_groups)} trajectory groups")
print(f"\nEach group contains trajectories for the same scenario:")

for i, group in enumerate(trajectory_groups[:2], 1):  # Show first 2 groups
    print(f"\nGroup {i}:")
    print(f"  Scenario: {group.trajectories[0].messages_and_choices[1]['content'][:60]}...")
    print(f"  Trajectories: {len(group.trajectories)}")
    print(f"  Rewards: {[f'{t.reward:.2f}' for t in group.trajectories]}")
    print(f"  Quality levels: {[t.metrics.get('quality') for t in group.trajectories]}")

print("\n" + "=" * 60)
print("RULER will rank trajectories within each group")
print("Higher-ranked trajectories get higher rewards for training")


Trajectory Groups for RULER
Created 3 trajectory groups

Each group contains trajectories for the same scenario:

Group 1:
  Scenario: What is machine learning?...
  Trajectories: 3
  Rewards: ['0.85', '0.60', '0.30']
  Quality levels: [None, None, None]

Group 2:
  Scenario: How does neural network training work?...
  Trajectories: 3
  Rewards: ['0.85', '0.60', '0.30']
  Quality levels: [None, None, None]

RULER will rank trajectories within each group
Higher-ranked trajectories get higher rewards for training


## Summary: Choosing the Right Approach

| Approach | Best For | Pros | Cons |
|----------|----------|------|------|
| **Automated (MCP)** | Well-defined tool sets | Diverse, scalable, tool-aware | Requires MCP server |
| **Manual** | Prototyping, golden examples | Precise control, specific cases | Time-consuming, limited scale |
| **Synthetic** | Large datasets, pattern coverage | Fast generation, systematic | May lack realism |
| **Real-World** | Production systems | Authentic, captures edge cases | Requires deployed system |
| **Hybrid** | Production training | Combines benefits of all | More complex setup |

### Key Takeaways:

1. **Start with automated generation** if you have MCP tools - it's the fastest way to get diverse scenarios
2. **Add manual examples** for specific behaviors you want to teach
3. **Use synthetic generation** to scale up and ensure coverage
4. **Collect real-world data** to improve on actual usage patterns
5. **Group trajectories** by scenario for RULER to perform relative ranking

The goal is to create **multiple trajectories per scenario** so RULER can learn which behaviors are better through comparison.


In [19]:
# Summary of all generated trajectories
print("Trajectory Generation Summary")
print("=" * 60)
print(f"\nTotal trajectories created: {len(all_trajectories)}")
print(f"  - Manual: {len(manual_trajectories)}")
print(f"  - Synthetic: {len(synthetic_trajectories)}")
print(f"  - Real-world: {len(real_world_trajectories)}")
if mcp_scenarios:
    print(f"  - MCP scenarios available: {len(mcp_scenarios)}")

print(f"\nTrajectory groups: {len(trajectory_groups)}")
print(f"Average trajectories per group: {sum(len(g.trajectories) for g in trajectory_groups) / len(trajectory_groups):.1f}")

print("\n" + "=" * 60)
print("Next steps:")
print("1. Generate more trajectories using rollout functions")
print("2. Group trajectories by scenario")
print("3. Use RULER to rank trajectories within groups")
print("4. Train the model on ranked trajectories")
print("\nSee improved_mcp_rl_working.ipynb for a complete training example!")


Trajectory Generation Summary

Total trajectories created: 14
  - Manual: 2
  - Synthetic: 9
  - Real-world: 3
  - MCP scenarios available: 10

Trajectory groups: 3
Average trajectories per group: 3.0

Next steps:
1. Generate more trajectories using rollout functions
2. Group trajectories by scenario
3. Use RULER to rank trajectories within groups
4. Train the model on ranked trajectories

See improved_mcp_rl_working.ipynb for a complete training example!
